In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
dbutils.widgets.text('catalog_name','feature_dab')
catalog_name = dbutils.widgets.get('catalog_name')
table_name = f'{catalog_name}.brnz.cust_raw'

In [0]:
print(table_name)

In [0]:
cust_df = (
    spark.readStream.format('cloudfiles')
    .option('cloudFiles.format','csv')
    .option('header','true')
    .option('cloudFiles.schemaLocation','/Workspace/Users/jafferali7248@gmail.com/custrepo/custtrydab/src/files/schemas/')
    .option('cloudFiles.schemaEvolutionMode','addNewColumns')
    .option('cloudFiles.inferColumnTypes','true')
    .load('/Workspace/Users/jafferali7248@gmail.com/custrepo/custtrydab/src/files/csvs')
    .withColumn('file_path',col('_metadata.file_path'))
    .withColumn('file_name',col('_metadata.file_name'))
    .withColumn('file_load_time_src',col('_metadata.file_modification_time'))
    .withColumn('load_ts',current_timestamp())
)

In [0]:
cust_df.writeStream.format('delta').option('mergeSchema','true').option('checkpointLocation','/Workspace/Users/jafferali7248@gmail.com/custrepo/custtrydab/src/files/checkpoints').trigger(availableNow=True).table(table_name)

In [0]:
display(spark.sql(f'select * from {table_name}'))